# Employee Activity Anomaly Detection - Training
RHOAI + GPU optimized

## Install dependencies

In [ ]:
!pip install --upgrade pip
!pip install tensorflow scikit-learn pandas numpy
!pip install onnx==1.17.0 onnxruntime==1.19.2 tf2onnx==1.16.1
!pip install protobuf==5.28.3

## Check GPU availability

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("Num GPUs:", len(tf.config.list_physical_devices('GPU')))

if len(tf.config.list_physical_devices('GPU')) > 0:
    print("\n✓ GPU detected and ready to use!")
    gpu = tf.config.list_physical_devices('GPU')[0]
    print(f"  Device: {gpu}")
else:
    print("\n⚠ No GPU detected, using CPU")

## Import libraries

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import pickle
import time

from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score

from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from keras.callbacks import EarlyStopping, ModelCheckpoint

import tf2onnx
import onnx
import onnxruntime as rt

print("✓ All imports successful")

## Load and prepare data

In [ ]:
feature_indexes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
label_indexes = [11]

print("Loading datasets...")
df_train = pd.read_csv('../data/train.csv')
X_train = df_train.iloc[:, feature_indexes].values
y_train = df_train.iloc[:, label_indexes].values

df_val = pd.read_csv('../data/validate.csv')
X_val = df_val.iloc[:, feature_indexes].values
y_val = df_val.iloc[:, label_indexes].values

df_test = pd.read_csv('../data/test.csv')
X_test = df_test.iloc[:, feature_indexes].values
y_test = df_test.iloc[:, label_indexes].values

print(f"\nDataset sizes:")
print(f"  Training:   {len(X_train)} samples")
print(f"  Validation: {len(X_val)} samples")
print(f"  Test:       {len(X_test)} samples")
print(f"\nFeatures: {len(feature_indexes)}")

## Scale features

In [ ]:
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Path("../artifact").mkdir(parents=True, exist_ok=True)
with open("../artifact/test_data.pkl", "wb") as f:
    pickle.dump((X_test_scaled, y_test), f)
with open("../artifact/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✓ Scaling complete")

## Calculate class weights

In [ ]:
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train.ravel()
)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}

print(f"Class weights: {class_weights_dict}")
print(f"Normal/Anomaly distribution in training:")
print(f"  Normal:  {(y_train == 0).sum()} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"  Anomaly: {(y_train == 1).sum()} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")

## Build model

In [ ]:
print("Building neural network...")

model = Sequential([
    Dense(64, activation='relu', input_dim=len(feature_indexes)),
    Dropout(0.3),
    
    Dense(64),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),
    
    Dense(32),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.2),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

## Setup callbacks

In [ ]:
Path("../checkpoints").mkdir(parents=True, exist_ok=True)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    '../checkpoints/best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, checkpoint]
print("✓ Callbacks configured")

## Train model

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32

print(f"\nStarting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print("=" * 70)

start_time = time.time()

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time

print("=" * 70)
print(f"\n✓ Training complete!")
print(f"  Time: {training_time:.2f} seconds ({training_time/60:.1f} minutes)")
print(f"  Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"  Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

## Plot training history

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history.history['accuracy'], label='Training')
axes[0, 0].plot(history.history['val_accuracy'], label='Validation')
axes[0, 0].set_title('Model Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history.history['loss'], label='Training')
axes[0, 1].plot(history.history['val_loss'], label='Validation')
axes[0, 1].set_title('Model Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history.history['precision'], label='Training')
axes[1, 0].plot(history.history['val_precision'], label='Validation')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history.history['recall'], label='Training')
axes[1, 1].plot(history.history['val_recall'], label='Validation')
axes[1, 1].set_title('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Save model to ONNX

In [ ]:
print("Converting model to ONNX format...")

@tf.function(input_signature=[tf.TensorSpec([None, len(feature_indexes)], tf.float32, name='dense_input')])
def model_fn(x):
    return model(x)

model_proto, _ = tf2onnx.convert.from_function(
    model_fn,
    input_signature=[tf.TensorSpec([None, len(feature_indexes)], tf.float32, name='dense_input')]
)

os.makedirs("../models/employee_anomaly/1", exist_ok=True)
onnx.save(model_proto, "../models/employee_anomaly/1/model.onnx")

print("✓ Model saved to: ../models/employee_anomaly/1/model.onnx")

## Verify saved model

In [ ]:
!ls -lh ../models/employee_anomaly/1/

## Evaluate on test set

In [ ]:
print("Loading ONNX model for inference...")
sess = rt.InferenceSession(
    "../models/employee_anomaly/1/model.onnx",
    providers=rt.get_available_providers()
)
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

print(f"  Input: {input_name}")
print(f"  Output: {output_name}")

In [ ]:
print("Running predictions on test set...")
y_pred_proba = sess.run(
    [output_name],
    {input_name: X_test_scaled.astype(np.float32)}
)[0]
y_pred_proba = np.squeeze(y_pred_proba)

threshold = 0.5
y_pred = (y_pred_proba > threshold).astype(int)

y_test_flat = y_test.ravel()

In [ ]:
accuracy = (y_pred == y_test_flat).sum() / len(y_test_flat)
precision = precision_score(y_test_flat, y_pred, zero_division=0)
recall = recall_score(y_test_flat, y_pred, zero_division=0)
f1 = f1_score(y_test_flat, y_pred, zero_division=0)

print("\n" + "="*70)
print("TEST SET EVALUATION RESULTS")
print("="*70)
print(f"  Accuracy:  {accuracy*100:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print("="*70)

In [ ]:
cm = confusion_matrix(y_test_flat, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomaly'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Test Set')
plt.show()

print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives:  {cm[0,0]}")
print(f"  False Positives: {cm[0,1]}")
print(f"  False Negatives: {cm[1,0]}")
print(f"  True Positives:  {cm[1,1]}")

## Test with examples

In [ ]:
def predict_activity(features, description):
    scaled = scaler.transform([features]).astype(np.float32)
    prob = sess.run([output_name], {input_name: scaled})[0][0][0]
    is_anomaly = prob > threshold
    status = "🚨 ANOMALY" if is_anomaly else "✅ Normal"
    
    print(f"\n{description}")
    print(f"  Probability: {prob:.4f}")
    print(f"  Prediction: {status}")
    return prob, is_anomaly

print("\n" + "="*70)
print("PREDICTION EXAMPLES")
print("="*70)

predict_activity(
    [192, 168, 1, 45, 10, 0, 0, 1, 0, 0, 0],
    "Normal: Office login at 10 AM"
)

predict_activity(
    [85, 232, 45, 100, 23, 1, 0, 0, 3, 1, 0],
    "Suspicious: Night download from external IP"
)

predict_activity(
    [10, 10, 10, 10, 14, 0, 0, 1, 5, 0, 1],
    "Suspicious: Multiple failed logins"
)

predict_activity(
    [192, 168, 1, 78, 14, 0, 0, 1, 2, 1, 0],
    "Normal: Document read during work hours"
)

predict_activity(
    [203, 0, 113, 50, 2, 1, 1, 0, 3, 1, 0],
    "Suspicious: Weekend + Night + External IP + Download"
)

print("\n" + "="*70)

## Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)
print(f"Model: Deep Neural Network (3 hidden layers)")
print(f"Training time: {training_time:.2f}s")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")
print(f"Test accuracy: {accuracy*100:.2f}%")
print(f"Test F1-score: {f1:.4f}")
print(f"\nModel saved: ../models/employee_anomaly/1/model.onnx")
print(f"Ready for deployment to RHOAI Model Serving")
print("="*70)